In [1]:
import sys
from pathlib import Path
from itertools import product

import pandas as pd

from xgboost import XGBRegressor
from sklearn.feature_selection import RFE
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Locate the repo root without importing from src yet.
_current = Path.cwd().resolve()
PROJECT_ROOT = next(
    candidate
    for candidate in (_current, *_current.parents)
    if (candidate / "AGENTS.md").exists()
)
sys.path.insert(0, str(PROJECT_ROOT))

from src.feature_selection.data_loading import load_split


In [2]:
TARGET = "target_return_5d"

X_train, y_train = load_split("train", processed_dir=PROJECT_ROOT / "data" / "processed")
X_valid, y_valid = load_split("validation", processed_dir=PROJECT_ROOT / "data" / "processed")

features = list(X_train.columns)

print("X_train:", X_train.shape)
print("X_valid:", X_valid.shape)
print("Feature count:", len(features))


X_train: (1895, 25)
X_valid: (600, 25)
Feature count: 25


Sanity check kept from the original notebook: confirm `return_5d` (past 5-day momentum) and `target_return_5d` (forward 5-day return) are computed from opposite time windows, not the same values.

In [3]:
sanity_df = pd.DataFrame({
    "return_5d": X_train["return_5d"],
    "target_return_5d": y_train,
})

print(sanity_df.head(10))
print("return_5d null:", X_train["return_5d"].isna().sum())
print("target_return_5d null:", y_train.isna().sum())


   return_5d  target_return_5d
0   0.094233          0.103529
1   0.022642          0.044280
2  -0.006605          0.061170
3  -0.014476         -0.019976
4  -0.025959         -0.019699
5   0.112261          0.086047
6   0.042471          0.061111
7   0.015936          0.061438
8  -0.047511         -0.002969
9  -0.034753         -0.005807
return_5d null: 0
target_return_5d null: 0


In [4]:
# RFE fit on TRAIN ONLY, for each (n_features_to_select, step) config.
rfe_feature_counts = [5, 10, 15, 20, 25]
rfe_steps = [1, 0.1]

rfe_results = []

for n_features in rfe_feature_counts:
    for step in rfe_steps:

        rfe_estimator = XGBRegressor(
            n_estimators=300,
            max_depth=3,
            learning_rate=0.1,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42,
            objective="reg:squarederror",
        )

        rfe = RFE(
            estimator=rfe_estimator,
            n_features_to_select=n_features,
            step=step,
        )

        rfe.fit(X_train, y_train)

        selected_features = X_train.columns[rfe.support_].tolist()

        rfe_results.append({
            "n_features_to_select": n_features,
            "step": step,
            "selected_features": selected_features,
        })

        print(f"n_features={n_features}, step={step} -> {selected_features}")


n_features=5, step=1 -> ['sma_5', 'sma_60', 'price_to_sma_60', 'volatility_20', 'atr_14']
n_features=5, step=0.1 -> ['sma_5', 'sma_60', 'macd_hist', 'volatility_20', 'atr_14']
n_features=10, step=1 -> ['sma_5', 'sma_60', 'price_to_sma_20', 'price_to_sma_60', 'macd', 'macd_signal', 'macd_hist', 'volatility_20', 'atr_14', 'volume_sma_20']
n_features=10, step=0.1 -> ['sma_5', 'sma_20', 'sma_60', 'price_to_sma_60', 'rsi_14', 'macd', 'macd_hist', 'volatility_20', 'atr_14', 'volume_sma_20']
n_features=15, step=1 -> ['return_10d', 'return_20d', 'sma_5', 'sma_20', 'sma_60', 'price_to_sma_20', 'price_to_sma_60', 'rsi_14', 'macd', 'macd_signal', 'macd_hist', 'volatility_20', 'atr_14', 'volume_sma_20', 'volume_ratio_20']
n_features=15, step=0.1 -> ['return_10d', 'return_20d', 'sma_5', 'sma_20', 'sma_60', 'price_to_sma_20', 'price_to_sma_60', 'rsi_14', 'roc_10', 'macd', 'macd_signal', 'macd_hist', 'volatility_20', 'atr_14', 'volume_sma_20']
n_features=20, step=1 -> ['return_5d', 'return_10d', 'ret

In [5]:
rfe_results_df = pd.DataFrame(rfe_results)

FEATURE_SETS_OUTPUT_DIR = PROJECT_ROOT / "data" / "processed" / "wrapper_results"
FEATURE_SETS_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

rfe_results_df.to_csv(
    FEATURE_SETS_OUTPUT_DIR / "rfe_xgboost_feature_sets.csv",
    index=False,
)


In [6]:
xgb_param_grid = list(product(
    [100, 300, 500],   # n_estimators
    [2, 3, 5],         # max_depth
    [0.03, 0.1],       # learning_rate
    [0.8, 1.0],        # subsample
    [0.8, 1.0],        # colsample_bytree
))

print("XGBoost combinations:", len(xgb_param_grid))
print("Total experiments:", len(rfe_results) * len(xgb_param_grid))


XGBoost combinations: 72
Total experiments: 720


In [7]:
results = []

total = len(rfe_results) * len(xgb_param_grid)
experiment = 0

for rfe_result in rfe_results:

    n_features = rfe_result["n_features_to_select"]
    step = rfe_result["step"]
    selected_features = rfe_result["selected_features"]

    X_tr = X_train[selected_features]
    X_va = X_valid[selected_features]

    for params in xgb_param_grid:

        experiment += 1
        n_estimators, max_depth, learning_rate, subsample, colsample_bytree = params

        model = XGBRegressor(
            n_estimators=n_estimators,
            max_depth=max_depth,
            learning_rate=learning_rate,
            subsample=subsample,
            colsample_bytree=colsample_bytree,
            random_state=42,
            objective="reg:squarederror",
        )

        model.fit(X_tr, y_train)

        y_pred = model.predict(X_va)

        mse = mean_squared_error(y_valid, y_pred)

        results.append({
            "rfe_n_features": n_features,
            "rfe_step": step,
            "selected_features": selected_features,
            "n_estimators": n_estimators,
            "max_depth": max_depth,
            "learning_rate": learning_rate,
            "subsample": subsample,
            "colsample_bytree": colsample_bytree,
            "rmse": mse ** 0.5,
            "mae": mean_absolute_error(y_valid, y_pred),
            "r2": r2_score(y_valid, y_pred),
        })

        if experiment % 50 == 0:
            print(f"[{experiment}/{total}] done")

results_df = pd.DataFrame(results)
print(results_df.shape)


[50/720] done
[100/720] done
[150/720] done
[200/720] done
[250/720] done
[300/720] done
[350/720] done
[400/720] done
[450/720] done
[500/720] done
[550/720] done
[600/720] done
[650/720] done
[700/720] done
(720, 11)


In [8]:
results_df.sort_values("rmse").head(10)


,rfe_n_features,rfe_step,selected_features,n_estimators,max_depth,learning_rate,subsample,colsample_bytree,rmse,mae,r2
75,5,0.1,"[sma_5, sma_60, macd_hist, volatility_20, atr_14]",100,2,0.03,1.0,1.0,0.100359,0.074859,0.080899
73,5,0.1,"[sma_5, sma_60, macd_hist, volatility_20, atr_14]",100,2,0.03,0.8,1.0,0.100545,0.074699,0.077481
74,5,0.1,"[sma_5, sma_60, macd_hist, volatility_20, atr_14]",100,2,0.03,1.0,0.8,0.100628,0.075057,0.075959
3,5,1.0,"[sma_5, sma_60, price_to_sma_60, volatility_20...",100,2,0.03,1.0,1.0,0.100827,0.074979,0.072312
72,5,0.1,"[sma_5, sma_60, macd_hist, volatility_20, atr_14]",100,2,0.03,0.8,0.8,0.100850,0.075127,0.071881
77,5,0.1,"[sma_5, sma_60, macd_hist, volatility_20, atr_14]",100,2,0.10,0.8,1.0,0.100872,0.075415,0.071472
219,10,0.1,"[sma_5, sma_20, sma_60, price_to_sma_60, rsi_1...",100,2,0.03,1.0,1.0,0.100916,0.074930,0.070676
1,5,1.0,"[sma_5, sma_60, price_to_sma_60, volatility_20...",100,2,0.03,0.8,1.0,0.100918,0.074905,0.070633
218,10,0.1,"[sma_5, sma_20, sma_60, price_to_sma_60, rsi_1...",100,2,0.03,1.0,0.8,0.100943,0.074954,0.070164
97,5,0.1,"[sma_5, sma_60, macd_hist, volatility_20, atr_14]",300,2,0.03,0.8,1.0,0.100975,0.075564,0.069590


In [9]:
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed" / "wrapper_results"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

output_path = OUTPUT_DIR / "rfe_xgboost_results.csv"
results_df.to_csv(output_path, index=False)

print("Saved:", output_path, results_df.shape)


Saved: /Users/yangjaehoon/Desktop/StockLens/data/processed/wrapper_results/rfe_xgboost_results.csv (720, 11)


In [10]:
results_df.groupby("rfe_step")["rmse"].min().sort_values()


rfe_step
0.1    0.100359
1.0    0.100827
Name: rmse, dtype: float64